# 多因子选股系统 - 探索性分析

本 notebook 展示如何使用多因子选股系统进行因子分析、回测和绩效评估。

## 1. 环境准备

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 导入系统模块
from config import load_config
from data import DataDownloader, DataProcessor
from factors import FactorEngine
from analysis import ICAnalyzer, Neutralizer
from backtest import BacktestEngine, PortfolioBuilder
from evaluation import PerformanceMetrics, ReportGenerator

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('环境准备完成')

## 2. 加载配置

In [ ]:
# 加载配置
config = load_config('../config/config.yaml')
print('配置加载完成')
print(f"数据源: {config.get('data_source.provider')}")
print(f"回测期间: {config.get('backtest.start_date')} 至 {config.get('backtest.end_date')}")
print(f"持仓数量: {config.get('portfolio.top_n')}")

## 3. 数据获取

In [ ]:
# 获取股票数据
downloader = DataDownloader(config)

# 获取沪深300成分股
symbols = downloader.get_index_components('000300.SH')
print(f"获取到 {len(symbols)} 只股票")

# 下载历史数据
stock_data = downloader.get_bulk_daily_data(
    symbols=symbols[:50],  # 取前50只作为演示
    start_date='20180101',
    end_date='20231231'
)
print(f"数据下载完成，共 {len(stock_data)} 条记录")

## 4. 数据处理

In [ ]:
# 数据处理
processor = DataProcessor(config)
processed_data = processor.process(stock_data)

# 查看数据
processed_data.head()

## 5. 因子计算

In [ ]:
# 创建因子引擎
factor_engine = FactorEngine(config)

# 获取所有可用因子
factor_info = factor_engine.get_factor_info()
print("可用因子列表:")
factor_info

In [ ]:
# 计算因子
# 这里选择几个核心因子进行演示
selected_factors = ['momentum_1m', 'volatility_20d', 'turnover_rate']

# 计算技术因子
from factors.technical import TechnicalFactors
tech_factors = TechnicalFactors()

factors_dict = {}
for factor_name in selected_factors:
    factor = tech_factors.get_factor_by_name(factor_name)
    if factor:
        factors_dict[factor_name] = factor.calculate(processed_data)

factors_df = pd.DataFrame(factors_dict)
print(f"因子计算完成，共 {len(factors_df)} 条记录")

## 6. IC 分析

In [ ]:
# IC 分析
ic_analyzer = ICAnalyzer(config)

# 准备收益率数据
returns = processed_data.set_index(['date', 'code'])['daily_return']

# 计算 IC
ic_results = ic_analyzer.calculate_ic(factors_df, returns)

print(f"平均 IC: {ic_results['mean_ic']:.4f}")
print(f"IC_IR: {ic_results['ir']:.4f}")

In [ ]:
# IC 统计
ic_stats = ic_results['ic_stats']
ic_stats

## 7. 回测

In [ ]:
# 创建回测引擎
portfolio_builder = PortfolioBuilder(config)
backtest_engine = BacktestEngine(config, portfolio_builder)

# 运行回测
results = backtest_engine.run(factors_df, processed_data)
print(f"回测完成，最终净值: {results.get('final_nav', 1):.4f}")

## 8. 绩效评估

In [ ]:
# 计算绩效指标
metrics = PerformanceMetrics(results, config)
perf = metrics.calculate()

# 显示摘要
summary = metrics.get_summary()
summary

## 9. 生成报告

In [ ]:
# 生成可视化报告
report_gen = ReportGenerator(config)
report_gen.plot_nav_curve(results)
report_gen.plot_drawdown(results)
report_gen.plot_returns_distribution(results)

print("报告已生成到 output 目录")

## 10. 下一步

1. 尝试不同的因子组合
2. 调整回测参数
3. 添加更多因子类型
4. 优化组合权重